<!-- quarto cv template: https://github.com/mps9506/quarto-cv -->


In [ ]:
#text_size_text = "\\footnotesize "
#text_size_text = "\\small "
text_size_text = "\\normalsize "

In [ ]:
import yaml
from IPython.display import display, Markdown, display_latex
from datetime import datetime

<!---

# Updated  <!-- need to have this line due to issue with quarto-cv-pdf templating -->

# Academic Appointment

\hrule

**South Dakota State University** \hfill Brookings, South Dakota, USA  
Assistant Professor, Department of Mathematics and Statistics \hfill 2024 to present  


# Education and Training  

\hrule

**University of Michigan** \hfill Ann Arbor, Michigan, USA  
Postdoctoral Research Fellow, Department of Internal Medicine ([NIH T32 HL007853](https://reporter.nih.gov/search/edYVbhxf0Uu_nQ9-JqHDJA/project-details/10576923)) \hfill 2023 to 2024  
Postdoctoral Research Fellow, Department of Biostatistics \hfill 2021 to 2023  

**University of Massachusetts Medical School** \hfill Worcester, Massachusetts, USA  
Postdoctoral Research Fellow, Department of Microbiology & Physiological Systems \hfill 2019 to 2021  

**University of Wisconsin-Madison** \hfill Madison, Wisconsin, USA  
Postdoctoral Research Fellow, Department of Biostatistics and Medical Informatics \hfill 2019     
Ph.D., Department of Statistics \hfill 2011 to 2019  
Postdoctoral Research Fellow ([NIH T32 ES007015](https://reporter.nih.gov/search/QFqt1UAzrUORCW6sbBz0qw/project-details/8296307)), Molecular and Environmental Toxicology Center \hfill 2012 to 2014  
Assistant Researcher, [Value-Added Research Center](https://varc.wceruw.org/) \hfill 2011  


**University of Washington** \hfill Seattle, Washington, USA   
Postdoctoral Research Fellow, Department of Biostatistics \hfill 2007 to 2009   

**University of Wisconsin-Madison** \hfill Madison, Wisconsin, USA  
M.D., School of Medicine and Public Health \hfill 2003 to 2007  
M.S., Department of Population Health Sciences \hfill 2003 to 2007  
Graduate Project Assistant, Department of Population Health Sciences \hfill 2003   

[**La Clinica de los Campesinos**](https://famhealth.org/) \hfill Wautoma, Wisconsin, USA  
Spanish Language Medical Interpreter \hfill 2002   

**University of Wisconsin-Madison** \hfill Madison, Wisconsin, USA  
B.S., with Honors in Chemistry, Mathematics, and Biochemistry \hfill 1996 to 2001    






# Written Works 

\hrule


In [ ]:
# we define ref_type in the main qmd file
import yaml
import datetime
from IPython.display import display, Markdown, display_latex

from datetime import datetime, date


def date_constructor(loader, node):
    value = loader.construct_scalar(node)

    # Adjust the format string based on your date format
    if isinstance(value, int):
        return date(value, 1, 1)
    elif "-" in value:
        if len(value) == 7:
            return datetime.strptime(value + "-01", "%Y-%m-%d").date()
        elif len(value) == 10:
            return datetime.strptime(value, "%Y-%m-%d").date()

    raise ValueError(f"Invalid date format: {value}")


yaml.add_constructor("tag:yaml.org,2002:date", date_constructor, yaml.SafeLoader)


with open("data/boehm-fixed.yaml", "r") as file:
    data = yaml.load(file, Loader=yaml.SafeLoader)

n_references = len(data["references"])
bibs = data["references"]

# define issued_formatted key for sorting
for article in bibs:
    if isinstance(article["issued"], int):
        article["issued_formatted"] = date(article["issued"], 1, 1)
    if isinstance(article["issued"], str):
        if len(article["issued"]) == 7:
            article["issued_formatted"] = datetime.strptime(
                article["issued"] + "-01", "%Y-%m-%d"
            ).date()
        elif len(article["issued"]) == 10:
            article["issued_formatted"] = datetime.strptime(
                article["issued"], "%Y-%m-%d"
            ).date()
    if isinstance(article["issued"], date):
        article["issued_formatted"] = article["issued"]


sorted_bibs = sorted(bibs, key=lambda x: x["issued_formatted"], reverse=True)


def render_with_hyperlink(value, url=None):
    if url is None:
        url = value
    markdown_text = f"[{value}]({url})"
    return markdown_text


# define a function for taking a bunch of names, in yaml, and producing a single string for printing


def assemble_names_from_yaml(names_yaml, my_name=["Boehm", "Boehm, III", "Boehm III"]):
    # Function body
    n_names = len(names_yaml)
    for i in range(n_names):
        name_value = names_yaml[i]
        if "literal" in name_value:
            name = name_value["literal"]
            if "Boehm" in name: #problematic only if other Boehms are in hte authors list
                name = f"\\textbf{{" + {text_size_text} + {name} + f"}}"
        else:
            name_last = name_value["family"]
            name_first = name_value["given"]
            name = name_first + " " + name_last
            #name = f"\\footnotesize " + name
            name = text_size_text + name
            if name_last in my_name:
                name = f"\\textbf{{" + name + f"}}"
        if i == 0:
            names = name
        elif i == n_names - 1:
            if n_names == 2:
                #names = names + f"\\footnotesize \ and " + name
                #names = names + text_size_text + f"\ and " + name
                names = names + text_size_text + r"\ and " + name
            else:
                #names = names + f"\\footnotesize , and " + name
                names = names + text_size_text + f", and " + name
        else:
            #names = names + f"\\footnotesize , " + name
            names = names + text_size_text + f", " + name
    # Code goes here
    return names  # You can return a value if needed


markdown_output = ""
mo_article = ""
mo_conference = ""
mo_software = ""
mo_manuscript = ""
mo_thesis = ""
mo_book = ""
mo_chapter = ""
mo_other = ""
# define counters
c_article = 0
c_conference = 0
c_software = 0
c_manuscript = 0
c_thesis = 0
c_book = 0
c_chapter = 0
c_other = 0




for ref_number in range(n_references):
    ref = sorted_bibs[ref_number]
    ref_type = ref["type"]
    # define formatted_text in this if statement
    title = ref["title"]
    # organize authors for printing
    author_yaml = ref["author"]
    authors = assemble_names_from_yaml(author_yaml)
    # check for editors in yaml
    if "editor" in ref:
        editor_yaml = ref["editor"]
        editors = assemble_names_from_yaml(editor_yaml)
    if ref_type == "paper-conference":
        c_conference += 1   
        date = ref["issued"]
        # publisher = ref["publisher"]
        proceedings = ref["container-title"]
        #formatted_text = f"\\footnotesize {authors} ({date}). {title}. In *{proceedings}*.  \n"  # , {publisher}."
        formatted_text = text_size_text + f"{authors} ({date}). {title}. In *{proceedings}*.  \n"  # , {publisher}."
        mo_conference += formatted_text
        # ie, if the citation is of the type we want to print right now, here
    if ref_type == "article-journal":  # need to have correct keys for each type!!
        c_article += 1
        # organize other info for printing.
        # ADD DOI AND VOLUME & ISSUE TO JOURNAL CITATIONS
        date = ref["issued"]
        doi = ref["doi"]
        journal = ref["container-title"]
        if "volume" in ref:
            volume = ref["volume"]
            if "issue" in ref:
                issue = ref["issue"]
                volume_issue = f"{volume}({issue})"
            else:
                volume_issue = f"{volume}"
        else:
            volume_issue = ""
        page = ref["page"]
        #formatted_text = f"\\footnotesize {authors} ({date}). {title}. *{journal}* {volume_issue}:{page}. " + render_with_hyperlink("https://doi.org/" + doi) + ".  \n"
        formatted_text = text_size_text + f"{authors} ({date}). {title}. *{journal}* {volume_issue}:{page}. " + render_with_hyperlink("https://doi.org/" + doi) + ".  \n"
        mo_article += formatted_text
        # Print the formatted content as Markdown
    if ref_type == "software":
        c_software += 1
        date = ref["issued"]
        url = ref["url"]
        #formatted_text = f"\\footnotesize {authors} ({date}). {title}. " + render_with_hyperlink(url) + ".  \n"
        formatted_text = text_size_text + f"{authors} ({date}). {title}. " + render_with_hyperlink(url) + ".  \n"
        mo_software += formatted_text
    # discard manuscripts that are merely "in preparation"
    # need to manually indicate in yaml file which manuscripts are submitted
    if ref_type == "manuscript":
        if ref["manuscript_type"] == "submitted":
            c_manuscript += 1
            date = "submitted"
            #formatted_text = f"\\footnotesize {authors} ({date}). {title}.  \n"
        if ref["manuscript_type"] == "in preparation":
            c_manuscript += 1
            date = "in preparation"
        formatted_text = text_size_text + f"{authors} ({date}). {title}.  \n"
        mo_manuscript += formatted_text
    if ref_type == "thesis":
        c_thesis += 1
        date = ref["issued"]
        #formatted_text = f"\\footnotesize {authors} ({date}). {title}.  \n"
        formatted_text = text_size_text + f"{authors} ({date}). {title}.  \n"
        mo_thesis += formatted_text
    if ref_type == "book":
        c_book += 1
        date = ref["issued"]
        authors = authors + ", editors"
        publisher = ref["publisher"]
        #formatted_text = f"\\footnotesize {authors} ({date}). *{title}*. {publisher}.  \n"
        formatted_text = text_size_text + f"{authors} ({date}). *{title}*. {publisher}.  \n"
        mo_book += formatted_text
    if ref_type == "chapter":
        c_chapter += 1
        date = ref["issued"]
        book_title = ref["container-title"]
        pages = ref["page"]
        publisher = ref["publisher"]
        editors = editors + ", editors"
        #formatted_text = f"\\footnotesize {authors} ({date}). {title}. In {editors} *{book_title}*. {publisher}.  \n"
        formatted_text = text_size_text + f"{authors} ({date}). {title}. In {editors} *{book_title}*. {publisher}.  \n"
        mo_chapter += formatted_text
    if ref_type == "other":
        c_other += 1
        date = ref["issued"]
        journal = ref["container-title"]
   #     formatted_text = f"\\footnotesize {authors} ({date}). {title}. *{journal}*.  \n"
        formatted_text = text_size_text + f"{authors} ({date}). {title}. *{journal}*.  \n"
        mo_other += formatted_text

In [ ]:
# order: manuscript, article, book, chapter, software, conference, thesis, other
manuscript_r = range(1, c_manuscript + 1)
article_r = range(1 + c_manuscript, 1 + c_manuscript + c_article)
book_r = range(1 + c_manuscript + c_article, 1 + c_manuscript + c_article + c_book)
chapter_r = range(1 + c_manuscript + c_article + c_book , 1 + c_manuscript + c_article + c_book + c_chapter)
software_r =  range(1 + c_manuscript + c_article + c_book + c_chapter, 1 + c_manuscript + c_article + c_book + c_chapter + c_software)
conference_r =  range(1 + c_manuscript + c_article + c_book + c_chapter + c_software + 0, 1 + c_manuscript + c_article + c_book + c_chapter + c_software + c_conference)
thesis_r =  range(1 + c_manuscript + c_article + c_book + c_chapter + c_software + c_conference + 0, 1 + c_manuscript + c_article + c_book + c_chapter + c_software + c_conference + c_thesis)
other_r = range(1 + c_manuscript + c_article + c_book + c_chapter + c_software + c_conference + c_thesis +  0, 1 + c_manuscript + c_article + c_book + c_chapter + c_software + c_conference + c_thesis + c_other)

In [ ]:
### CHatGPT ###
#myrange = range(1, 4)
#mystring = "This is a multiline\nstring\nwith line breaks."

# Convert the range to a list and add the numbers, period, and space
#formatted_strings = [f"{num}. " + line for num, line in zip(myrange, mystring.split('\n'))]

# Join the formatted strings with newline characters
#result = '\n'.join(formatted_strings)

#print(result)
# insert numbers into strings!
#mo_manuscript = f"\\footnotesize **" + mo_manuscript
fmo_manuscript = [f"**{num}.** " + line for num, line in zip(manuscript_r, mo_manuscript.split('\n'))]
fmo_article = [f"**{num}.** " + line for num, line in zip(article_r, mo_article.split('\n'))]
fmo_conference = [f"**{num}.** " + line for num, line in zip(conference_r, mo_conference.split('\n'))]
fmo_software = [f"**{num}.** " + line for num, line in zip(software_r, mo_software.split('\n'))]
fmo_thesis = [f"**{num}.** " + line for num, line in zip(thesis_r, mo_thesis.split('\n'))]
fmo_book = [f"**{num}.** " + line for num, line in zip(book_r, mo_book.split('\n'))]
fmo_chapter = [f"**{num}.** " + line for num, line in zip(chapter_r, mo_chapter.split('\n'))]
fmo_other = [f"**{num}.** " + line for num, line in zip(other_r, mo_other.split('\n'))]

## Unpublished Manuscripts


In [ ]:
display(Markdown('\n'.join(fmo_manuscript)))

## Journal Articles 


In [ ]:
display(Markdown('\n'.join(fmo_article)))

## Edited Book  


In [ ]:
display(Markdown('\n'.join(fmo_book)))

## Book Chapters


In [ ]:
display(Markdown('\n'.join(fmo_chapter)))

## Software


In [ ]:
display(Markdown('\n'.join(fmo_software)))

## Conference Proceedings


In [ ]:
display(Markdown('\n'.join(fmo_conference)))

## Ph.D. Thesis


In [ ]:
display(Markdown('\n'.join(fmo_thesis)))

## Other Publications


In [ ]:
display(Markdown('\n'.join(fmo_other)))

# Teaching Experience

\hrule


In [ ]:
# define a "" for every course type: formal, short
markdown_output_short = ""
markdown_output_formal = ""

In [ ]:
# use anthropic's claude chat bot
import sys
#print(sys.version)

import datetime
import yaml
from IPython.display import display, Markdown, display_latex

with open("data/teaching.yaml", "r") as file:
    data = yaml.load(file, Loader=yaml.SafeLoader)

courses = data["courses"]

def convert_term_to_datetime(term):
    try:
        # Try parsing the term as a month and year
        return datetime.datetime.strptime(term, "%B %Y")
    except ValueError:
        # If it fails, try parsing as a season and year
        month_mapping = {"Spring": 4, "Summer": 7, "Fall": 9, "Winter": 1}
        season, year = term.split(" ")
        month = month_mapping.get(season, 1)  # Default to January if not found
        return datetime.datetime(int(year), month, 1)

# Assuming your dictionary of courses is called 'courses'
sorted_courses = []

# Convert term strings to datetime objects and create tuples
for course in courses:
    terms = course['term']
    sorted_terms = sorted([convert_term_to_datetime(term) for term in terms])
    sorted_courses.append((sorted_terms[0], course))  

# Sort the tuples by the datetime objects in reverse order
sorted_courses.sort(reverse=True, key=lambda x: x[0])

# Now 'sorted_courses' contains your courses in reverse chronological order
#for term_datetime, course in sorted_courses:
#    print(", ".join(course['term']), course['title'])

In [ ]:
for course in sorted_courses:
    cc = course[1]
    foo = ""
    terms = cc.get("term", [])
    t_formatted = ", ".join(terms) # join (possibly) multiple terms into a single string
    if cc.get("url") is not None:
        #foo = f"\\footnotesize {course['role']}, [{course['title']}, {course['institution']}]({course['url']})  \hfill " + f"\\footnotesize {t_formatted}  \n"
        foo = text_size_text + f"{cc['role']}, [{cc['title']}, {cc['institution']}]({cc['url']})  \\hfill " + f"{t_formatted}  \n"
    else:
        #foo = f"\\footnotesize {course['role']}, {course['title']}, {course['institution']}  \hfill " + f"\\footnotesize {t_formatted}  \n"
        foo = text_size_text + f"{cc['role']}, {cc['title']}, {cc['institution']}  \\hfill " + f"{t_formatted}  \n"
    if cc.get("type") == "short":
        markdown_output_short += foo
    elif cc.get("type") == "formal":
        markdown_output_formal += foo

## Formal Courses


In [ ]:
display(Markdown(markdown_output_formal))

## Short Courses & Workshops


In [ ]:
display(Markdown(markdown_output_short))

## Training in Teaching  


In [ ]:
# Read YAML file
with open("data/teacher-training.yaml", "r") as file:
    data = yaml.safe_load(file)

In [ ]:
markdown_output = ""

for tcourse in data['courses']:
    #foo = f"\\footnotesize {tcourse['title']}, {tcourse['institution']} \hfill {tcourse['term']}  \n"
    foo = text_size_text + f"{tcourse['title']}, {tcourse['institution']} \\hfill {tcourse['term']}  \n"
    markdown_output += foo
display(Markdown(markdown_output))

# Mentoring Experience

\hrule


In [ ]:
# Read YAML file
with open("data/mentoring.yaml", "r") as file:
    data = yaml.safe_load(file)

# Select elements

In [ ]:
markdown_output = ""

for person in data['people']:
    #foo = f"\\footnotesize Research mentor to {person['name']}, {person['title']} \hfill {person['start']} to {person['end']}   \nProject: {person['project']}   \nNext position: {person['next_role']}  \n"
    foo = text_size_text + f"Research mentor to {person['name']}, {person['title']} \\hfill {person['start']} to {person['end']}   \nProject: {person['project']}   \nNext position: {person['next_role']}  \n"
    markdown_output += foo
display(Markdown(markdown_output))

# Leadership Training

\hrule


In [ ]:
# Read YAML file
with open("data/leadership-training.yaml", "r") as file:
    data = yaml.safe_load(file)

In [ ]:
# maybe add more fields to output??? ie, in call to Markdown?
# need to decide which fields matter when assembling my yaml file!
markdown_output = ""
for event in data['events']:
    foo = ""
    if event.get("date") is not None:
        date_str = event.get("date")
        date_obj = datetime.datetime.strptime(date_str, "%Y-%m")
        formatted_date = date_obj.strftime("%B %Y")
        event['term'] = formatted_date
            # handle end_date
    end_date_str =  event.get("end_date")
    if end_date_str is not None:
        end_date_obj = datetime.datetime.strptime(end_date_str, "%Y-%m")
        end_date_formatted = end_date_obj.strftime("%B %Y")
        formatted_date = f"{formatted_date} to {end_date_formatted}"
        event['term'] = formatted_date
    if event.get("url") is not None:
        url = event.get("url")
        #foo = f"\\footnotesize [{event['role']}, {event['event']}]({url}) \hfill {event['term']}  \n"
        foo = text_size_text + f"[{event['role']}, {event['event']}]({url}) \\hfill {event['term']}  \n"
    else:
   #     foo = f"\\footnotesize {event['role']}, {event['event']} \hfill {event['term']}  \n"
        foo = text_size_text + f"{event['role']}, {event['event']} \\hfill {event['term']}  \n"
    markdown_output += foo
display(Markdown(markdown_output))

# Awards & Honors

\hrule


In [ ]:
# Read YAML file
with open("data/awards.yaml", "r") as file:
    data = yaml.safe_load(file)
sorted_awards = sorted(data.get("awards", []), key=lambda x: x.get("year", 0), reverse=True)
markdown_output = ""
for award in sorted_awards:
    year = award.get("year")
    end_year = award.get("end_year")
    if end_year is not None:
        year = f"{year} to {end_year}"
    #markdown_output += f"\\footnotesize {award['title']}, {award['institution']} \hfill {year}  \n"
    markdown_output += text_size_text + f"{award['title']}, {award['institution']} \\hfill {year}  \n"
display(Markdown(markdown_output))

# Presentations

\hrule


In [ ]:
# Read YAML file
with open("data/presentations.yaml", "r") as file:
    data = yaml.safe_load(file)

sorted_presentations = sorted(data.get("presentations", []), key=lambda x: x.get("date"), reverse=True)

markdown_output_talks = ""
markdown_output_posters = ""

for presentation in sorted_presentations:
    foo = ""
    date_str = presentation.get("date")
    date_obj = datetime.datetime.strptime(date_str, "%Y-%m-%d")
    date_formatted = date_obj.strftime("%B %d, %Y")
    url = presentation.get("url")
    if url is not None:
#        foo = f"\\footnotesize [{presentation['title']}]({url}) \hfill {date_formatted}  \n{presentation['venue']} \hfill {presentation['location']}  \n"
        foo = text_size_text + f"[{presentation['title']}]({url}) \\hfill {date_formatted}  \n{presentation['venue']} \\hfill {presentation['location']}  \n"
    else:
        #foo = f"\\footnotesize {presentation['title']} \hfill {date_formatted}  \n{presentation['venue']} \hfill {presentation['location']}  \n"
        foo = text_size_text + f"{presentation['title']} \\hfill {date_formatted}  \n{presentation['venue']} \\hfill {presentation['location']}  \n"
    if presentation.get("presentation_type") == "talk":
        markdown_output_talks += foo
    elif presentation.get("presentation_type") == "poster":
        markdown_output_posters += foo

## Slide Presentations


In [ ]:
display(Markdown(markdown_output_talks))

## Poster Presentations


In [ ]:
display(Markdown(markdown_output_posters))

# Service


\hrule


In [ ]:
# Read YAML file
with open("data/service.yaml", "r") as file:
    data = yaml.safe_load(file)
sorted_service = sorted(data.get("service", []), key=lambda x: max(x.get("date")), reverse=True)

markdown_output_university = ""
markdown_output_community = ""
markdown_output_professional = ""


for service in sorted_service:
    foo = ""
    # Accessing and formatting the dates as "Month Year"
    dates = service.get("date", [])
    formatted_dates = []
    for date_str in dates:
        date_obj = datetime.datetime.strptime(date_str, "%Y-%m")
        formatted_date = date_obj.strftime("%B %Y")
        # handle end_date
        end_date_str =  service.get("end_date")
        if end_date_str is not None:
            end_date_obj = datetime.datetime.strptime(end_date_str, "%Y-%m")
            end_date_formatted = end_date_obj.strftime("%B %Y")
            formatted_date = f"{formatted_date} to {end_date_formatted}"
        # handle current values
        if service.get("current") == 'true':
            formatted_date = f"{formatted_date} to present"
        formatted_dates.append(formatted_date)

    fd = ", ".join(formatted_dates)
    
    # handle those entries with multiple orgs
    organizations = service.get("organization", [])
    org_output = ", ".join(organizations)
    # handle those entries with url values
    url = service.get("url")
    if url is not None:
        #foo = f"\\footnotesize [{service['title']}]({url}), {org_output} \hfill {fd}  \n"
        foo = text_size_text + f"[{service['title']}]({url}), {org_output} \\hfill {fd}  \n"
    else:
        #foo = f"\\footnotesize {service['title']}, {org_output} \hfill {fd}  \n"
        foo = text_size_text + f"{service['title']}, {org_output} \\hfill {fd}  \n"
    if service.get("service_type") == "university":
        markdown_output_university += foo
    elif service.get("service_type") == "community":
        markdown_output_community += foo
    elif service.get("service_type") == "professional":
        markdown_output_professional += foo

## University Service


In [ ]:
display(Markdown(markdown_output_university))

## Community Service


In [ ]:
display(Markdown(markdown_output_community))

## Professional Service


In [ ]:
display(Markdown(markdown_output_professional))